# Lecture 23 - Reasoning Large Language Models

[![View notebook on Github](https://img.shields.io/static/v1.svg?logo=github&label=Repo&message=View%20On%20Github&color=lightgrey)](https://github.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/blob/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/Lecture_23-Reasoning_LLMs.ipynb)
[![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avakanski/Fall-2026-Applied-Data-Science-with-Python/blob/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/Lecture_23-Reasoning_LLMs.ipynb)
[![View on Website](https://img.shields.io/static/v1.svg?logo=quarto&label=Web&message=View%20on%20Website&color=blue)](https://avakanski.github.io/Fall-2026-Applied-Data-Science-with-Python/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/Lecture_23-Reasoning_LLMs.html)

<a id='top'></a>

- [23.1 Introduction to Reasoning Models](#23.1-introduction-to-reasoning-models)
    - [23.1.1 Test-time Compute](#23.1.1-test-time-compute)
    - [23.1.2 Main Categories of Reasoning Models](#23.1.2-main-categories-of-reasoning-models)
- [23.2 Inference-based Reasoning Models](#23.2-inference-based-reasoning-models)
    - [23.2.1 Chain-of-Thought Prompting](#23.2.1-chain-of-thought-prompting)
    - [23.2.2 Self-Consistency Sampling](#23.2.2-self-consistency-sampling)
- [23.3 Supervised Finetuning Reasoning Models](#23.3-supervised-finetuning-reasoning-models)
- [23.4 Reinforcement Learning-based Reasoning Models](#23.4-reinforcement-learning-based-reasoning-models)
    - [23.4.1 Outcome and Process Reward Models](#23.4.1-outcome-and-process-reward-models)
    - [23.4.2 RL Algorithms for Reasoning Models](#23.4.2-rl-algorithms-for-reasoning-models)
    - [23.4.3 Finetuning a Reasoning Model using GRPO](#23.4.3-finetuning-a-reasoning-model-using-grpo)
- [23.5 Reasoning Models Limitations](#23.5-reasoning-models-limitations)
- [References](#references)

<a id="23.1-introduction-to-reasoning-models"></a>

## 23.1 Introduction to Reasoning Models

Compared to regular LLMs that immediately produce the answer to a given question, **reasoning LLMs** break down a problem into smaller steps called **reasoning steps** or **thought processes** before answering.

<img src="https://raw.githubusercontent.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/images/regular_vs_reasoning_llms.png" width="600">

*Figure: Regular vs Reasoning LLMs.* Source [1].

I.e., instead of directly outputting the final answer, a reasoning LLM first produces a series of intermediate steps that explain how the model arrived at the conclusion. These steps make the internal processing more transparent and easier to follow. Reasoning steps are often referred to as **chain-of-thought (CoT)**, or in some sources they are also called *reasoning chain* or *reasoning traces* or *reasoning trajectory*.

<img src="https://raw.githubusercontent.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/images/reasoning_steps.png" width="500">

*Figure: Reasoning steps to produce the answer.* Source [1].

LLM developers describe the reasoning process as the model spending more time “thinking” through the problem before responding. However, it is important to note that reasoning LLMs do not actually think or reason in the human sense, and these terms are used more for convenience. LLMs generate responses autoregressively, one token at a time, based on statistical patterns learned from training data. The same applies to reasoning steps: they are also generated token-by-token from learned patterns. Consequently, there is no guarantee that LLM-generated reasoning steps are logical or correct.

Producing intermediate steps often improves LLM performance, and  has led to major advancements in generating solutions to complex reasoning tasks. This caused an important shift among LLM developers to focus on reasoning models. Today, most premier LLMs are equipped with reasoning abilities, and when answering users' questions they commonly display that they are “thinking” through the problem.

<a id="23.1.1-test-time-compute"></a>

### 23.1.1 Test-time Compute

Until 2024, developers typically improved LLM performance during pre-training and finetuning steps by collecting larger datasets (increasing the number of tokens), designing larger models (increasing the number of parameters), and using parallel computing across many GPUs (increasing the number of FLOPs). This approach is referred to as **train-time compute**, since increases across the three dimensions—dataset size, model size, and computing power—occur during model training.

In this context, the term *"compute"* refers to the computational resources required to train or run a model. Compute is typically expressed in floating-point operations (FLOPs), which measure the number of performed mathematical operations (e.g., multiplications, additions, etc).  

The relationship between compute scale and model performance is described by **scaling laws**. Scaling laws diagrams are usually shown on a log scale to illustrate how model quality improves with increased compute. An example is presented in the next figure. In the figure, `pass@1 accuracy` is a performance metric that indicates how often the model solves a problem correctly on the first attempt. It measures how reliable the model is without sampling multiple attempts. E.g., `pass@k accuracy` is a related metric that involves multiple attempts and measures how often the correct answer appears in any of k generated samples.

Well-known scaling laws include the Kaplan and Chinchilla laws, which imply that model performance increases with more data tokens, parameters, and compute FLOPs. The laws suggest that all three factors must be scaled simultaneously for optimal performance.

In 2024, OpenAI released the o1 reasoning model and introduced **test-time compute** scaling curves, which are also informally called test-time scaling law. They demonstrated that increasing computation during inference can boost model performance similarly to increasing train-time compute. Test-time compute is also referred to as **inference-time compute**.

<img src="https://raw.githubusercontent.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/images/scaling_laws.jpg" width="600">

*Figure: LLM scaling laws.* Source [1].

Test-time compute scaling caused a paradigm shift in LLM development. Instead of focusing primarily on train-time scaling through pre-training and finetuning, recent LLMs use more compute during inference to achieve improved reasoning and enhanced performance.

The figure below illustrates the difference in test-time compute between a non-reasoning LLM (which uses one token during inference) and reasoning models that use 6 and 15 tokens, respectively, to generate answers to the same question. By using more compute during inference, a reasoning LLM derives the answer through step-by-step "thinking".

<img src="https://raw.githubusercontent.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/images/tokens_compute.png" width="900">

*Figure: Comparison of used tokens for generating an answer*. Source [1].

<a id="23.1.2-main-categories-of-reasoning-models"></a>

### 23.1.2 Main Categories of Reasoning Models

Numerous approaches have recently been introduced for enhancing LLM reasoning abilities. These methods can be classified into four main categories:

1. **Inference-based reasoning methods**: These methods improve reasoning at inference time by using prompting and sampling strategies, but without retraining the model. That is, model weights are fixed, and no architectural changes are made. Examples include:

    - *Chain-of-thought prompting* - prompting the model to produce step-by-step explanations.
    - *Self-consistency sampling* - sampling multiple reasoning paths and selecting the most frequent answer.
    - *Tree-of-thought* - exploring multiple reasoning branches using a search algorithm.
    - *Reflection decoding* - prompting the model to reflect on mistakes and iteratively revise its solution.

2. **Supervised finetuning reasoning models**: The models are finetuned on datasets containing reasoning steps. Unlike inference-based methods that don't change model weights, this category of methods updates model weights during finetuning. Reasoning datasets typically contain questions and corresponding chain-of-thought reasoning steps and final answers. The datasets are often created based on solved problems from math or logic domains. The model learns reasoning patterns directly from supervised data. An exemplary approach is *distillation-based reasoning*, where reasoning abilities are transferred from a large, powerful LLM to a smaller and more efficient LLM.

3. **Reinforcement learning-based reasoning models**: Employ Reinforcement Learning (RL) algorithms that encourage the model to select reasoning steps that maximize reward signals. Rewards can differ across tasks, e.g., they can be defined based on producing correct answers to math problems, quality of intermediate reasoning steps, etc. This approach allows the model to learn reasoning strategies through trial and error, based on whether the selected steps increase the reward value. Like supervised finetuning category of reasoning models, RL-based methods also apply additional finetuning and update the model weights.

4. **Modular or hybrid reasoning models**: This approach enhances model reasoning by using external modules or tools. For instance, LLMs can use calculators or search engines to answer specific questions, rely on retrieval-augmented reasoning techniques, or use other domain-specific tools.

<a id="23.2-inference-based-reasoning-models"></a>

## 23.2 Inference-based Reasoning Models

As explained above, **inference-based reasoning methods** rely on prompting and sampling techniques that improve the reasoning capabilities of LLMs without modifying the model's weights or changing the architecture. Although such models still generate responses in the standard autoregressive manner as any other text, certain prompting and sampling strategies can significantly enhance their reasoning performance. Reasons for the improvements include:

- The model is encouraged to reason through intermediate steps before producing the final answer.
- Complex problems are decomposed into simpler subproblems that are easier to solve.
- Sampling multiple reasoning paths increases the likelihood of exploring correct solutions.
- The model can reflect on its own responses, identify mistakes, and self-correct its reasoning.


Inference-based reasoning techniques are simple to use because they require only prompt engineering or sampling multiple responses from a fixed model. The following sections provide examples of implementing basic inference-based reasoning methods using the Hugging Face library.

**Load Required Packages**

Let's first install the necessary Hugging Face packages `transformers`, `peft`, `trl` and `datasets`, and import the relevant modules. These steps are familiar from the previous lectures in this course.

In [1]:
# Install packages
!pip install -qq transformers datasets trl peft

In [2]:
# Import libraries and modules
import torch
# Causal Language Model and Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, logging
# Supress warnings
logging.set_verbosity_error()
# Parameter-efficient fine-tuning
from peft import LoraConfig, get_peft_model, TaskType
# Dataset handling
from datasets import load_dataset
# GRPO training components
from trl import GRPOConfig, GRPOTrainer
# Regex patterns for reward functions
import re
# Counter for tracking number of occurrences
from collections import Counter

**Load Model and Tokenizer**

For this demonstration, we will use a fairly small LLM `Qwen3-0.6B-Base`. This model has been pre-trained for general language understanding. The suffix `Base` indicates that the model has not been instruction-tuned or finetuned for reasoning tasks.

In [3]:
# Non-reasoning model
non_reasoning_model_name = "Qwen/Qwen3-0.6B-Base"

# Load the model
model = AutoModelForCausalLM.from_pretrained(
    non_reasoning_model_name,
    # Auto-distribute across available GPUs/CPU
    device_map="auto",
    # Allow custom model code execution
    trust_remote_code=True,
    # Use FP16 to reduce memory
    dtype=torch.float16,
    # Optimize memory usage during loading
    low_cpu_mem_usage=True,
)

# Load corresponding tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    non_reasoning_model_name,
    trust_remote_code=True
)
# Ensure tokenizer has proper padding token for batch processing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

**Prompt**

Let's now evaluate the model's answer to the following question. Although it is a simple math problem, LLMs often fail to generate a correct answer because they are trained primarily for next-token prediction rather than for solving logical or mathematical problems.

In [4]:
prompt = (r"Half the value of $3x-9$ is $x+37$. What is the value of $x$?")

This cell formats the question by adding instructions before the question. Since this is a base model, we will pass plain text rather than a chat-formatted system message. The instructions require the model to write the result on a new line in an answer box as `\boxed{ANSWER}`. The correct answer to the provided question is 83.

In [5]:
prompt_fmt = ("You are a helpful math assistant.\n"
        "Answer the question and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:")

In [6]:
print(prompt_fmt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:


**Generate Response**

The code in the next cell first converts the text in the prompt into a sequence of tokens, then the model generates a response in the form of output tokens, which afterward are decoded into text.

For response generation, we use greedy decoding with `do_sample=False`, which always selects the most likely next token, so that the result is reproducible. In this case, the LLM provided the final answer as 20, which is incorrect. This illustrates the limitation of standard autoregressive decoding.

In [7]:
input_tokens = tokenizer(prompt_fmt, return_tensors="pt").to(model.device)

output_tokens = model.generate(
        **input_tokens,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

output_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print(output_text)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer: \boxed{20}


<a id="23.2.1-chain-of-thought-prompting"></a>

### 23.2.1 Chain-of-Thought Prompting

Next, let's apply CoT prompting to encourage the LLM to reason step by step. A simple way to achieve this is by modifying the original prompt via CoT prompting and append the phrase "Explain your response step by step." Since this phrase is the only difference from the previous prompt, any change in the answer is due to CoT prompting.

Notice in the output of the next cell that the model provides the intermediate steps in solving the problem, as Steps 1 to 3. This reasoning process likely helped generate the correct final answer 83.

We can also notice that the output of CoT prompting is much longer than the direct answer in the previous case where the LLM produced only a few tokens for the final answer. With CoT prompting, the model describes in detail all steps for yielding the final answer. Hence, this approach requires much larger computational resources, and it is significantly more expensive.

In [8]:
prompt_chainofthought = prompt_fmt + " Explain your response step by step."

In [9]:
input_tokens = tokenizer(prompt_chainofthought, return_tensors="pt").to(model.device)

output_tokens = model.generate(
        **input_tokens,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

output_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print(output_text)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer: Explain your response step by step. To solve the problem, let's break it down step by step.

**Given:**
- Half the value of \( 3x - 9 \) is \( x + 37 \).

**Step 1: Translate the given statement into an equation.**

Half the value of \( 3x - 9 \) means \( \frac{1}{2} \times (3x - 9) \). According to the problem, this is equal to \( x + 37 \).

So, the equation is:
\[
\frac{1}{2} \times (3x - 9) = x + 37
\]

**Step 2: Eliminate the fraction by multiplying both sides by 2.**

\[
2 \times \left( \frac{1}{2} \times (3x - 9) \right) = 2 \times (x + 37)
\]

Simplifying both sides:
\[
3x - 9 = 2x + 74
\]

**Step 3: Solve for \( x \).**

Subtract \( 2x \) from both sides to get:
\[
3x - 2x - 9 = 74
\]
\[
x - 9 = 74
\]

Add 9 to both sides:
\[
x = 74 + 9
\]
\[
x = 83
\]

**Final Answer:**
\[
\boxed{83}

<a id="23.2.2-self-consistency-sampling"></a>

### 23.2.2 Self-Consistency Sampling

Besides generating longer outputs with more tokens as in the case of CoT prompting, another inference-based approach involves generating multiple responses, each with their own reasoning paths. Afterward, an aggregation method is applied to select a single final answer.

**Self-consistency sampling** is one such strategy that samples multiple responses, and selects the most frequent answer as the final answer. Because of the adopted aggregation, this technique is also called *majority voting*.

The process of generating multiple responses is also referred to as *parallel decoding* as multiple decoding paths are explored.  

<img src="https://raw.githubusercontent.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/images/self_consistency.png" width="600">

*Figure: Self-consistency sampling.* Source [1].

An example is presented next, where the model generates 5 reasoning paths. To increase reasoning diversity, we used `temperature` and `top_p` sampling. The answer of each path is extracted as the last number in the response. In this case, 4 out of the 5 paths produced 83 as an answer, which is therefore selected as the final answer. Note that this simple extraction is not perfect: in Path 3 it also captured the period at the end of the sentence ("83."), and Path 2 did not contain any number, so no answer was extracted.

In [10]:
# Generate multiple reasoning paths
num_paths = 5

# List to save all answers
all_answers = []

for path_id in range(num_paths):
    input_tokens = tokenizer(prompt_chainofthought, return_tensors="pt").to(model.device)
    output_tokens = model.generate(
        **input_tokens,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    output_text = tokenizer.decode(output_tokens[0][input_tokens['input_ids'].shape[1]:], skip_special_tokens=True)

    # Extract numeric answer for this path
    numbers = re.findall(r'\d+\.?\d*', output_text)
    extracted_answer = numbers[-1] if numbers else None

    # Append the answer for the current path
    all_answers.append(extracted_answer)

    print(f"\nPath {path_id + 1}:")
    print(f"Extracted Answer: {extracted_answer}")


Path 1:
Extracted Answer: 83

Path 2:
Extracted Answer: None

Path 3:
Extracted Answer: 83.

Path 4:
Extracted Answer: 83

Path 5:
Extracted Answer: 83


In [11]:
# Majority vote to find the most common answer
answer_counts = Counter(all_answers)
most_common_answer, _ = answer_counts.most_common(1)[0]

print(f"\n Self-Consistency Answer: {most_common_answer}")


 Self-Consistency Answer: 83


In [12]:
# Delete non-reasoning model and tokenizer from GPU memory
del model
del tokenizer
torch.cuda.empty_cache()

The main benefits of inference-based reasoning approaches are that they are simple, effective, and do not require retraining the model. On the other hand, these techniques do not provide the model with new knowledge as they simply change how the model uses its existing knowledge to generate more accurate and reliable responses.

In addition, although inference-based techniques can improve performance via CoT, they do not guarantee that the generated answers are correct. The model may still produce flawed reasoning, and it can introduce unnecessary steps that lead to incorrect conclusions.

<a id="23.3-supervised-finetuning-reasoning-models"></a>

## 23.3 Supervised Finetuning Reasoning Models

**Supervised finetuning (SFT) reasoning models** are developed by finetuning a pre-trained model using supervised learning on a dataset comprising questions and CoT outputs. An advantage of supervised finetuning reasoning over inference-based approaches is that the model weights are updated during finetuning, which usually improves the reasoning capabilities.

The quality of the training data is a critical factor in this approach. On the other hand, manually creating high-quality reasoning datasets is a labor-intensive and expensive process. It requires experts to compose step-by-step solutions that undergo rigorous verification. Recent advances now allow large reasoning models to generate synthetic CoT examples for supervised finetuning. Several open-source datasets containing solved math and logic problems are available and have been broadly used for this purpose.

As discussed in the previous section, *knowledge distillation* can be used to finetune smaller models using reasoning traces generated by larger reasoning models. This technique allows small LLMs to inherit reasoning abilities from large LLMs.

The SFT strategy for creating reasoning models is straightforward to implement and it is similar to standard instruction-following tuning procedures we discussed earlier in the course in the LLM lecture. I.e., SFT reasoning models use the standard cross-entropy loss for parameter update, and the models learn to predict the next token in a reasoning CoT, resulting in stable and predictable performance.

However, SFT reasoning models have also disadvantages compared to inference-based reasoning models. As previously stated, they require large, carefully curated datasets of high-quality reasoning traces, as well as they require computational resources and finetuning pipelines to update the model weights.

The most important limitation of SFT reasoning models is that they tend to memorize the form of reasoning in the finetuning examples rather than learning the underlying logic. In other words, these models learn how to reason through problems that are similar to those in the dataset for supervised finetuning, but their reasoning performance can degrade on unseen or different problems.

<a id="23.4-reinforcement-learning-based-reasoning-models"></a>

## 23.4 Reinforcement Learning-based Reasoning Models

**RL-based reasoning models** use reward signals to evaluate the quality of their intermediate reasoning steps or final answers, allowing the model to improve and refine the reasoning process. This approach enables the model to explore different reasoning paths during finetuning and gradually select strategies that yield higher rewards.

Unlike SFT reasoning models, which are finetuned by minimizing a cross-entropy loss, RL-based reasoning models are finetuned to maximize a reward signal. Because the model is rewarded for outcomes rather than for imitating given reasoning traces, RL-based finetuning tends to generalize better than SFT to problems that are not similar to the examples in the finetuning dataset. Reward maximization also encourages RL models to develop more efficient search strategies over possible solutions.

RL-driven reasoning techniques are currently among the most powerful methods for improving the reasoning capabilities of modern LLMs. They often outperform inference-based or supervised finetuning models on complex tasks that require mathematical or logical reasoning.

Despite the advantages, RL reasoning models are more difficult to finetune and are more sensitive to hyperparameters. On the other hand, an important advantage of RL with verifiable rewards is that it does not require datasets with written reasoning steps, and instead pairs of questions and final answers are sufficient, because the model generates its own reasoning and only the outcome is checked. RL models can also suffer from “reward hacking,” where the model finds shortcuts to maximize rewards but produces incorrect CoT reasoning.

It is also important to differentiate RL-based optimization for reasoning from Reinforcement Learning with Human Feedback (RLHF), which we studied in the lecture on LLMs. In particular, RLHF is used during the alignment phase of LLM development (also called preference tuning). In RLHF, rewards are produced by a reward model that is trained on data from human annotators who rank or compare model responses, and the objective of RLHF is to guide the model toward generating safe outputs that are aligned with human preferences. Conversely, in reasoning models, rewards for RL are computed automatically based on whether the model produces the correct final answer to a given question, or whether it produces correct intermediate reasoning steps.

<a id="23.4.1-outcome-and-process-reward-models"></a>

### 23.4.1 Outcome and Process Reward Models

Applying RL to develop reasoning LLMs relies on **verifiable tasks** where the correctness of an answer or a reasoning step can be automatically checked. Typical examples include math problems, logic puzzles, programming tasks, and scientific questions where solutions can be validated. For instance, math problems can be checked by comparing the model's answer to the ground-truth solution, and programming tasks can be verified by running the generated code to see if it passes all test cases. The verification results serve as reward signals for training reasoning models with RL. Because verification in these domains can be done automatically, they are well suited for developing reasoning models, since verifiable tasks allow the model to explore many reasoning paths and receive reward signals for its performance.

Based on how rewards are assigned in RL-based reasoning models, the models can be classified into Outcome Reward Models (ORM) and Process Reward Models (PRMs).

**Outcome Reward Models (ORM)** assign rewards based only on the correctness of the final answer. Although the model may produce a full CoT, only the final output is checked to determine whether the reasoning path was successful. If the answer is correct, the entire path is rewarded; otherwise, it receives little or no reward. A limitation of ORM is that it does not evaluate the quality of intermediate reasoning steps, so the model must independently discover strategies that lead to correct final answers.

**Process Reward Models (PRM)** extend this idea by evaluating the intermediate steps leading to the final answer. I.e., PRMs assign rewards to individual reasoning steps, e.g., partial derivations in a math solution or intermediate code sections in a programming task. This helps the model learn more structured reasoning, and makes it easier to avoid hallucinations or logically inconsistent CoT. PRMs are especially useful for tasks that require multi-step reasoning, because they provide feedback on each step that leads to correct reasoning rather than only correct outcomes. On the other hand, PRMs are more expensive to create, since training them requires labels for individual reasoning steps.

<img src="https://raw.githubusercontent.com/avakanski/Fall-2026-Applied-Data-Science-with-Python/main/Lectures/Theme_3-Model_Engineering/Lecture_23-Reasoning_LLMs/images/orm_prm.png" width="800">

*Figure: Outcome Reward Models vs Process Reward Models.* Source [1].

<a id="23.4.2-rl-algorithms-for-reasoning-models"></a>

### 23.4.2 RL Algorithms for Reasoning Models

Various RL algorithms have been proposed that use reward signals assigned by ORMs or PRMs for updating LLM parameters to improve a model's ability to "think" through complex problems.

Among the initial RL algorithms used for reasoning LLMs is **PPO (Proximal Policy Optimization)**. This algorithm uses actor and critic models, where the actor is the LLM being trained that generates the responses, and the critic is a separate model typically of similar size as the actor that estimates the expected reward. The critic's estimate is used to judge whether a generated response is better or worse than expected, and the actor is updated accordingly. However, PPO is computationally expensive, and its high memory requirements have driven the adoption of more efficient algorithms.

**GRPO (Group Relative Policy Optimization)** eliminates the need for a critic model and replaces it with statistical group averaging. For each question, GRPO samples a group of several responses and scores each response relative to the others in its group: the *advantage* of a response is its reward subtracted by the group's mean reward, and divided by the group's standard deviation. Responses that are better than the group average are reinforced, and responses that are worse are discouraged. This frees up GPU resources and allows the model to explore long CoT sequences.

Similar efficiency is achieved by **RLOO (REINFORCE Leave-One-Out)**, which also learns from multiple reasoning paths without a separate critic model.

Related approaches include bootstrapping frameworks like **STaR (Self-Taught Reasoner)** and **Iterative DPO** (Iterative Direct Preference Optimization), in which the model iteratively learns from its own successes, teaching itself to reason better over time.

Please note that the detailed mathematical internal workings of these algorithms are outside the scope of this course.

<a id="23.4.3-finetuning-a-reasoning-model-using-grpo"></a>

### 23.4.3 Finetuning a Reasoning Model using GRPO

This section demonstrates using GRPO (Group Relative Policy Optimization) RL algorithm to finetune an LLM for mathematical reasoning. We will use the GSM8K dataset that contains a curated collection of approximately 8,000 solved math problems. GSM8K is commonly used as a benchmark dataset for reasoning models.

The objective is for the model to generate structured mathematical solutions and provide the intermediate reasoning steps for solving the problems.

#### Load Dataset

The next cell loads separately the train and test splits of the GSM8K mathematical reasoning dataset.

In [13]:
# Load train split of GSM8K
train_dataset = load_dataset("openai/gsm8k", "main", split="train")

# Load test split of GSM8K
test_dataset = load_dataset("openai/gsm8k", "main", split="test")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [14]:
# Check the dataset
train_dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})

In [15]:
# Check the first math problem
train_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [16]:
# Print the first question and answer
print("Question:", train_dataset[0]['question'])
print("Answer:", train_dataset[0]['answer'])

Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


GSM8K answers use a `####` marker before the final numeric answer. This next function extracts the final answer to create clean targets.

In [17]:
# Dataset processing utilities
def extract_hash_answer(text):
    """Extract numerical answer from GSM8K format (#### marker)"""
    if "####" not in text:
        return None
    # GSM8K uses format: "Answer... #### 72"
    return text.split("####")[1].strip()

Next, we define a short structured output format for training, consisting of a reasoning section and a solution section.

In [18]:
# Define structured output format for mathematical reasoning
reasoning_start = "<REASONING>"
reasoning_end = "</REASONING>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

# System prompt that defines the desired reasoning structure
system_prompt = f"""You are a mathematical reasoning assistant.
When given a math problem:
1. Show your step-by-step work between {reasoning_start} and {reasoning_end}
2. Provide ONLY your final numerical answer between {solution_start} and {solution_end}
   - Example: <SOLUTION> 18 </SOLUTION>
3. Be precise and show all calculation steps clearly in the reasoning section."""

This code formats dataset examples as conversations for GRPO.

In [19]:
def process_dataset_example(example):
    """Convert GSM8K example to conversation format for GRPO training"""
    question = example["question"]
    answer = extract_hash_answer(example["answer"])

    # Create conversation with system prompt for structured reasoning
    prompt = [{"role": "system", "content": system_prompt},
        {"role": "user", "content": question},]

    return {"prompt": prompt, "answer": answer}

In [20]:
# Apply conversation formatting to all examples
train_dataset = train_dataset.map(process_dataset_example)

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

In [21]:
# Print the first example
train_dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'content': 'You are a mathematical reasoning assistant.\nWhen given a math problem:\n1. Show your step-by-step work between <REASONING> and </REASONING>\n2. Provide ONLY your final numerical answer between <SOLUTION> and </SOLUTION>\n   - Example: <SOLUTION> 18 </SOLUTION>\n3. Be precise and show all calculation steps clearly in the reasoning section.',
   'role': 'system'},
  {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
   'role': 'user'}]}

#### Model Loading

For this task we will use `Llama-3.2-3B-Instruct` with 3B parameters.

In [22]:
# Select model
model_name = "unsloth/Llama-3.2-3B-Instruct"
# Token limit for mathematical problems
max_seq_length = 2048

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # Auto-distribute across available GPUs/CPU
    device_map="auto",
    # Allow custom model code execution
    trust_remote_code=True,
    # Use FP16 to reduce memory
    torch_dtype=torch.float16,
    # Optimize memory usage during loading
    low_cpu_mem_usage=True,
)

# Load corresponding tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Ensure tokenizer has proper padding token for batch processing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

We will apply LoRA (Low-Rank Adaptation) parameter-efficient finetuning to update only approximately 0.3% of the model parameters.

In [23]:
# Colab has torchao preinstalled, which is not used in this notebook, but causes a dependency error
# Therefore, we need to uninstall it
!pip uninstall -qq -y torchao

In [24]:
# Configure LoRA for mathematical reasoning adaptation
lora_config = LoraConfig(
    # Rank: adaptation capacity
    r=16,
    # Scaling factor (typically 2x rank)
    lora_alpha=32,
    # Focus on attention for reasoning
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    # Regularization to prevent overfitting
    lora_dropout=0.1,
    # Skip bias adaptation for simplicity
    bias="none",
    # Causal language modeling task
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA configuration to create trainable adapter
model = get_peft_model(model, lora_config)

# Enable gradient checkpointing to reduce memory usage
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Display trainable vs total parameters
model.print_trainable_parameters()

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


#### RL Reward Definition

This cell defines the reward signal for GRPO. The reward uses a combination of 4 functions that evaluate different aspects of the generated responses:

- Match Format Exactly: Assigns high reward if the format of the response matches the required response pattern.
- Match Format Approximately: Assigns partial reward if the response matches individual components of the response pattern, even when the matching is not perfect.
- Check Answer Correctness: Assigns rewards for mathematical accuracy with graduated scoring (e.g., 3.0 reward for exact match, 1.5 reward for answer within 10%, etc.).
- Check Numbers Extraction: Assign rewards if the model can parse the output and extract numerical results.

The adopted reward strategy is an Outcome Reward Model (ORM), since rewards are assigned for producing a correct final answer. However, the rewards are computed with simple rules rather than with a trained reward model. Furthermore, the strategy uses additional functions that assign rewards for the format of the response, as well as for approximate correctness of the generated response.

In [25]:
# The code in this cell is not required in preparation for quizzes

# Compiled regex patterns for efficient reward computation
match_format = re.compile(
rf"\A[\s]{{0,}}"                     # Optional whitespace at start
rf"{reasoning_start}.+?{reasoning_end}.*?"  # Reasoning section (non-greedy)
rf"{solution_start}(.+?){solution_end}"     # Solution section with capture group
rf"[\s]{{0,}}\Z",                    # Optional whitespace at end
flags=re.DOTALL                       # Matches newlines
)

match_numbers = re.compile(
    rf"{solution_start}.*?(-?\d+(?:,\d{{3}})*(?:\.\d+)?)", # Extract numbers from solution section
    flags=re.MULTILINE | re.DOTALL        # Flexible pattern matching
)

# Matches from solution_start until the closing tag OR the end of the string
match_lenient = re.compile(
    rf"{solution_start}\s*(.*?)(?:{solution_end}|\Z)",
    flags=re.DOTALL
)

# Reward Function 1: Exact Format Compliance
def match_format_exactly(completions, **kwargs):
    """
    High reward (3.0) for perfect format adherence
    Ensures model learns the complete structured output pattern
    """
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        # Check if response matches complete format pattern
        score = 3.0 if match_format.search(response) is not None else 0.0
        scores.append(score)
    return scores

# Reward Function 2: Partial Format Credit
def match_format_approximately(completions, **kwargs):
    """
    Graduated scoring for format elements
    Encourages learning individual components even if not perfect
    """
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        score = 0

        # Award +0.5 for correct token count, -0.5 for wrong count
        score += 0.5 if response.count(reasoning_start) == 1 else -0.5
        score += 0.5 if response.count(reasoning_end) == 1 else -0.5
        score += 0.5 if response.count(solution_start) == 1 else -0.5
        score += 0.5 if response.count(solution_end) == 1 else -0.5

        scores.append(score)
    return scores

# Reward Function 3: Mathematical Accuracy
def check_answer_correctness(prompts, completions, answer, **kwargs):
    """
    Graduated scoring for mathematical accuracy:
    - 3.0: Exact match
    - 1.5: Within 10% (close answer)
    - 0.5: Within 20% (reasonable attempt)
    - -0.5: Wrong answer (penalty for incorrect math)
    """
    responses = [completion[0]["content"] for completion in completions]

    # Extract answers using format pattern
    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:  # No extractable answer
            scores.append(0)
            continue

        # Remove commas and dollar signs, e.g. "$1,200" can be compared to "1200"
        guess = guess.replace(',', '').replace('$', '')
        true_answer = true_answer.replace(',', '')

        # Exact string match gets full points
        if guess.strip() == true_answer.strip():
            scores.append(3.0)
        else:
            # Try numerical comparison for partial credit
            try:
                ratio = float(guess) / float(true_answer)
                if 0.9 <= ratio <= 1.1:      # Within 10%
                    scores.append(1.5)
                elif 0.8 <= ratio <= 1.2:    # Within 20%
                    scores.append(0.5)
                else:                         # Wrong answer
                    scores.append(-0.5)
            except (ValueError, ZeroDivisionError):
                scores.append(-0.5)           # Invalid numerical format

    return scores

# Reward Function 4: Number Extraction Ability
def check_numbers_extraction(prompts, completions, answer, **kwargs):
    """
    Tests the model's ability to extract numerical values from solution sections
    Complementary to exact format matching - focuses on parsing capability
    """
    responses = [completion[0]["content"] for completion in completions]

    # Extract numbers from solution sections using number pattern
    extracted_responses = [
        guess.group(1) if (guess := match_numbers.search(r)) is not None else None
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:  # No extractable number
            scores.append(0)
            continue

        try:
            # Simple numerical equality check
            true_val = float(true_answer.strip())
            # Remove commas before converting
            guess_val = float(guess.strip().replace(',', ''))
            # Binary scoring: correct (1.5) or incorrect (0)
            scores.append(1.5 if guess_val == true_val else 0.0)
        except (ValueError, TypeError):
            scores.append(0)  # Invalid number format

    return scores

#### Baseline Evaluation Before Training

Before finetuning, let's evaluate the model on the first 50 problems from the test dataset, so that later we can measure the impact of GRPO training. The function `evaluate_model` formats each question with the chat template, generates the responses in batches, and extracts the numerical answer from the `<SOLUTION>` section. The full responses are printed for the first 5 problems.

In [26]:
# The code in this cell is not required in preparation for quizzes
def extract_model_answer(response):
    """Extract the final numerical answer from a model response"""
    # Use the exact format first; if it fails, take the text after the <SOLUTION> tag
    format_match = match_format.search(response)
    if format_match:
        solution_text = format_match.group(1)
    else:
        lenient_match = match_lenient.search(response)
        solution_text = lenient_match.group(1) if lenient_match else None
    if not solution_text:
        return None
    # Take the last number in the solution text, without commas or a trailing period
    numbers = re.findall(r'-?\d[\d,]*\.?\d*', solution_text)
    return numbers[-1].replace(',', '').rstrip('.') if numbers else None


def evaluate_model(model, dataset, num_problems=50, batch_size=10, num_to_print=5):
    """Evaluate the model on the first num_problems problems of the dataset using greedy decoding"""
    model.eval()
    correct_count = 0

    for start in range(0, num_problems, batch_size):
        batch = dataset.select(range(start, min(start + batch_size, num_problems)))

        # Format each question with the system prompt and the chat template
        texts = [tokenizer.apply_chat_template(
                     [{"role": "system", "content": system_prompt},
                      {"role": "user", "content": question}],
                     tokenize=False, add_generation_prompt=True)
                 for question in batch["question"]]

        # The chat template already contains the begin-of-text token, so don't add it a second time
        inputs = tokenizer(texts, return_tensors="pt", padding=True,
                           add_special_tokens=False).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                # Greedy decoding for reproducible results
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
            )
        responses = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:],
                                           skip_special_tokens=True)

        for i, (question, answer_text, response) in enumerate(
                zip(batch["question"], batch["answer"], responses)):
            problem_number = start + i + 1
            true_answer = extract_hash_answer(answer_text).replace(',', '')
            model_answer = extract_model_answer(response)

            # Check if correct
            try:
                is_correct = model_answer is not None and float(model_answer) == float(true_answer)
            except ValueError:
                is_correct = False
            correct_count += is_correct

            if problem_number <= num_to_print:
                print(f"Test Problem {problem_number}")
                print(f"Question: {question}")
                print(f"\nTrue Answer: {true_answer}")
                print(f"\nModel Response:\n{response}")
                print(f"\nExtracted Answer: {model_answer}")
                print("Correct" if is_correct else "Incorrect")
                print("=" * 80)

    accuracy = correct_count / num_problems
    print(f"Results: {correct_count}/{num_problems} correct ({accuracy*100:.0f}%)")
    return accuracy

In [27]:
# Evaluate the model before GRPO training
baseline_accuracy = evaluate_model(model, test_dataset)

Test Problem 1
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

True Answer: 18

Model Response:
<SOLUTION>

Janet's ducks lay 16 eggs per day. She eats 3 eggs for breakfast, so the number of eggs left is:
16 - 3 = 13

She bakes muffins with 4 eggs, so the number of eggs left for the farmers' market is:
13 - 4 = 9

She sells the remaining 9 eggs at the farmers' market for $2 per egg. To find the total amount she makes, multiply the number of eggs sold by the price per egg:
9 * $2 = $18

Extracted Answer: 18
Correct
Test Problem 2
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

True Answer: 3

Model Response:
<SOLUTION>

To find the total number of bolts needed, we need to calcula

In [28]:
print(f"\nAccuracy before GRPO training: {baseline_accuracy*100:.0f}%")


Accuracy before GRPO training: 64%


#### Configure GRPO Training Arguments

GRPO training arguments are defined next. GRPO trainer in Hugging Face has similar format to SFT trainer, except that it defines the reward functions (`reward_funcs` argument) for updating the model. To output the values of the rewards, we added the simple callback function `RewardLoggingCallback`.

 In GRPO, the batch size counts generated responses rather than questions. With `per_device_train_batch_size=2` and `gradient_accumulation_steps=8`, each training step processes 2 × 8 = 16 responses. Since `num_generations=4` this means that 4 responses are generated for each question, and the 16 responses in each step come from 16 / 4 = 4 questions. Hence, 100 training steps use 4 × 100 = 400 questions out of the 7,473 training questions in the dataset.

In [29]:
# Configure GRPO training parameters
training_args = GRPOConfig(
    # Learning rate
    learning_rate=5e-6,
    # Batch configuration
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    # Sequence length limits for mathematical problems
    max_completion_length=1024,
    # Training duration
    # num_train_epochs=1,
    max_steps = 100,
    # Log every 10 steps
    logging_steps=10,
    # Enable FP16 training for memory efficiency
    fp16=True,
    bf16=False,
    # Output configuration
    output_dir="./trl_grpo_outputs",
    # Gradient clipping for stable training
    max_grad_norm=0.1,
    # Disable external logging
    report_to=[],
    # Show the progress bar and loss table
    disable_tqdm=False,
    # Generation parameters for better reward signal
    num_generations=4,
    # Higher temp for more diverse outputs
    temperature=0.8,
    # KL divergence penalty for GRPO
    beta=0.01,
)

In [30]:
# Custom Trainer to log rewards
from transformers import TrainerCallback

class RewardLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "reward" in logs:
            print(f"Step {state.global_step:3d} | reward {float(logs['reward']):.2f} | "
                  f"format exact {float(logs['rewards/match_format_exactly/mean']):.2f} | "
                  f"format approx {float(logs['rewards/match_format_approximately/mean']):.2f} | "
                  f"correctness {float(logs['rewards/check_answer_correctness/mean']):.2f} | "
                  f"numbers {float(logs['rewards/check_numbers_extraction/mean']):.2f}")


trainer = GRPOTrainer(
    model=model,
    # Four complementary reward functions
    reward_funcs=[
        match_format_exactly,         # Structure compliance
        match_format_approximately,   # Partial format credit
        check_answer_correctness,     # Mathematical accuracy
        check_numbers_extraction,     # Number parsing ability
    ],
    # Training configuration
    args=training_args,
    # Processed GSM8K dataset
    train_dataset=train_dataset,
    # Tokenizer (processing_class in TRL)
    processing_class=tokenizer,
    # Log RL rewards
    callbacks=[RewardLoggingCallback()]
)

Note that the GRPO algorithm uses a loss function that is different than the cross-entropy loss that is commonly used in ANN classification and for supervised training or finetuning, and the GRPO loss value is not a useful measure of progress. As we stated earlier, GRPO compares each response to the other responses in its group, and the advantages (reward minus group mean, divided by group standard deviation) average to zero within every group. Consequently, the loss fluctuates around zero regardless of whether the rewards are high or low. The loss also includes a KL-divergence penalty weighted by `beta`, which keeps the finetuned model close to the original model, and with `beta=0.01`, this term is very small in our run.

A more relevant metric to monitor in GRPO is the reward values, where increasing rewards indicate improving behavior. As you recall, we defined the overall reward as a sum of four reward criteria. In the output of the training cell below, the combined reward is displayed under the, well, "reward" variable. During the training, the reward increased from 3.95 at step 10 to 8.27 at step 100.

Note that the maximum possible reward is 9.5 (3 + 2 + 3 + 1.5). More than half of the increase came from the format rewards. The reward for `format_exact` increased from 1.35 to 2.74 (out of 3), and `format_approx` from 0.79 to 1.82 (out of 2), whereas check answer `correctness` increased from 1.10 to 2.43 (out of 3). The correctness reward also depends on the format, since it is 0 when no answer can be extracted. I.e., in this short training run, the model mainly learned to follow the required output format.

In [31]:
trainer.train()

Step,Training Loss
10,0.051396
20,0.023463
30,0.048197
40,0.044918
50,0.009759
60,0.015728
70,0.044383
80,0.030251
90,0.014189
100,0.009377


Step  10 | reward 3.95 | format exact 1.35 | format approx 0.79 | correctness 1.10 | numbers 0.70
Step  20 | reward 6.08 | format exact 1.99 | format approx 1.41 | correctness 1.70 | numbers 0.99
Step  30 | reward 6.82 | format exact 2.27 | format approx 1.61 | correctness 1.85 | numbers 1.10
Step  40 | reward 7.63 | format exact 2.61 | format approx 1.73 | correctness 2.09 | numbers 1.20
Step  50 | reward 7.94 | format exact 2.59 | format approx 1.77 | correctness 2.28 | numbers 1.30
Step  60 | reward 7.81 | format exact 2.55 | format approx 1.75 | correctness 2.22 | numbers 1.29
Step  70 | reward 7.67 | format exact 2.62 | format approx 1.79 | correctness 2.12 | numbers 1.13
Step  80 | reward 8.67 | format exact 2.89 | format approx 1.90 | correctness 2.54 | numbers 1.34
Step  90 | reward 7.97 | format exact 2.64 | format approx 1.79 | correctness 2.30 | numbers 1.25
Step 100 | reward 8.27 | format exact 2.74 | format approx 1.82 | correctness 2.43 | numbers 1.27


TrainOutput(global_step=100, training_loss=0.029166108444333077, metrics={'train_runtime': 5268.0049, 'train_samples_per_second': 0.304, 'train_steps_per_second': 0.019, 'total_flos': 0.0, 'train_loss': 0.029166108444333077, 'epoch': 0.05353319057815846})

#### Evaluate on Test Set Problems

The finetuned model is next evaluated with the same `evaluate_model` function that we used before training, on the same first 50 problems from the test dataset, so the two accuracies can be compared directly. The full responses are printed for the first 5 problems.

In this case, the accuracy changed from 64% before training to 70% after training. This is a modest improvement and evaluating on more test problems would be needed to confirm it.

The effect of training on the output format is easier to see. Before training, the model skipped the reasoning section in 3 of the 5 printed responses (Problems 1, 2, and 4 start directly with the `<SOLUTION>` tag), whereas after training all 5 responses contain both the `<REASONING>` and `<SOLUTION>` sections.

Note that premier LLMs have achieved over 97% accuracy on the GSM8K dataset.  Meta reports about 78% accuracy for `Llama-3.2-3B-Instruct` on GSM8K (with 8-shot CoT prompting) [7], so the model solves many of these problems even before finetuning.

In [32]:
finetuned_accuracy = evaluate_model(model, test_dataset)

Test Problem 1
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

True Answer: 18

Model Response:
<REASONING>
To find out how much Janet makes every day at the farmers' market, we need to first calculate the total number of eggs she lays and then subtract the eggs she eats and bakes.

Janet lays 16 eggs per day.

She eats 3 eggs for breakfast every morning, so she eats 3 eggs/day.

She bakes 4 eggs for her friends every day, so she bakes 4 eggs/day.

Total eggs eaten and baked = 3 + 4 = 7 eggs/day

Now, let's subtract the eggs eaten and baked from the total eggs laid:
16 (eggs laid) - 7 (eggs eaten and baked) = 9 eggs/day

Janet sells the remaining 9 eggs at the farmers' market for $2 per egg.

To find the total amount she makes, multiply the numbe

In [33]:
print(f"\nAccuracy before GRPO training: {baseline_accuracy*100:.0f}%")
print(f"Accuracy after GRPO training:  {finetuned_accuracy*100:.0f}%")


Accuracy before GRPO training: 64%
Accuracy after GRPO training:  70%


<a id="23.5-reasoning-models-limitations"></a>

## 23.5 Reasoning Models Limitations

Despite the advantages, reasoning LLMs have several important limitations to consider.

- Overthinking: reasoning models “overthink,” i.e., since they can generate many more tokens than standard models to reach the same answer, in some settings they loop through unnecessarily long CoT instead of using external tools or stopping early. This makes them less efficient and sometimes less reliable.
- Limits of longer reasoning: more compute at test time does not always help. In some cases, longer CoT sequences actually hurt accuracy, where the model amplifies its own mistakes instead of fixing them. Some studies showed that reasoning models can fail to generalize on planning or low-complexity tasks where standard non-reasoning models do better.
- Degradation in non-reasoning domains: although finetuning a model for reasoning can improve math or code abilities, sometimes this can reduce performance on general instruction-following or other simpler tasks.
- Increased cost and latency: reasoning requires extra tokens, which increases compute cost, latency, and uses up the context window. For some tasks this extra cost is justified, while for others it is wasteful. These trade-offs are important when choosing or deploying models.

Reasoning models are most useful when a task truly requires multi-step thinking, such as solving math and logic problems, writing complex code, planning, or handling tasks where the answer must be derived rather than recalled. In these cases, the model benefits from exploring intermediate steps and evaluating different reasoning paths. However, reasoning is not always necessary or desirable. For simpler tasks like retrieving factual information,
summarization, or translation, standard LLMs are usually faster, cheaper, and often more accurate because they avoid unnecessary chains of thought. For instance, if the user asks "What is the capital of Italy?" there is no need to use a reasoning model to produce the answer. A good practical rule is to use the right tool for the task, and rely on reasoning models when genuine thinking is required, and use standard models when the task is straightforward.

## References

1. A Visual Guide to Reasoning LLMs, by Maarten Grootendorst, available at [https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-reasoning-llms](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-reasoning-llms).
2. Build A Reasoning Model (From Scratch), by Sebastian Raschka, Manning books, 2025, source code available at [https://github.com/rasbt/reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch).
3. Demystifying Reasoning Models, by Cameron R. Wolfe, available at [https://cameronrwolfe.substack.com/p/demystifying-reasoning-models](https://cameronrwolfe.substack.com/p/demystifying-reasoning-models).
4. Advanced GRPO Fine-tuning for Mathematical Reasoning with Multi-Reward Training, by Behrooz Azarkhalili, available at [https://huggingface.co/learn/cookbook/en/trl_grpo_reasoning_advanced_reward](https://huggingface.co/learn/cookbook/en/trl_grpo_reasoning_advanced_reward).
5. Unsloth Notebooks: Llama3_2_(3B)_GRPO_LoRA.ipynb, available at [https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Advanced_Llama3_2_(3B)_GRPO_LoRA.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Advanced_Llama3_2_(3B)_GRPO_LoRA.ipynb).
6. IBM: What is a reasoning model?, by Dave Bergmann, available at [https://www.ibm.com/think/topics/reasoning-model](https://www.ibm.com/think/topics/reasoning-model).
7. Llama 3.2 3B Instruct model card, by Meta, available at [https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct).


[BACK TO TOP](#top)